In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder \
    .appName("Pertemuan11-MLlibRegresi") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

print("SparkSession siap. Versi Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/12 15:58:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession siap. Versi Spark: 3.5.9


In [4]:
import numpy as np
import pandas as pd

np.random.seed(42)
n = 2000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo"]

data = pd.DataFrame({
    "unit_terjual": np.random.randint(1, 15, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "rating": np.random.randint(1, 6, size=n),
})
# Total pendapatan dipengaruhi unit & harga (dengan sedikit variasi acak, seperti data real world)
data["pendapatan"] = data["unit_terjual"] * data["harga_satuan"] + np.random.normal(0, 20000, size=n)

df = spark.createDataFrame(data)
df.show(5)
print("Total baris:", df.count())

+------------+------------+------------+----------+------+------------------+
|unit_terjual|harga_satuan|    kategori|      kota|rating|        pendapatan|
+------------+------------+------------+----------+------+------------------+
|           7|      150000|     Fashion|  Magelang|     4| 997913.0612957058|
|           4|      100000|Rumah Tangga|  Semarang|     4| 389490.8317158621|
|          13|      150000|  Elektronik|Yogyakarta|     4|1936047.6709878189|
|          11|      100000|Rumah Tangga|  Magelang|     5|1080918.9109179082|
|           8|      150000|     Makanan|  Semarang|     2|1191696.4943767258|
+------------+------------+------------+----------+------+------------------+
only showing top 5 rows



[Stage 1:>                                                          (0 + 4) / 4]

Total baris: 2000


In [5]:
from pyspark.ml.feature import StringIndexer

# StringIndexer mengubah nilai kategori (teks) menjadi indeks numerik
# berdasarkan frekuensi kemunculan (nilai paling sering muncul -> indeks 0, dst.)
indexer_kategori = StringIndexer(inputCol="kategori", outputCol="kategori_idx")
indexer_kota = StringIndexer(inputCol="kota", outputCol="kota_idx")

# Perhatikan pola dua tahap: fit() dulu (mempelajari daftar kategori), baru transform() (menerapkannya)
df_indexed = indexer_kategori.fit(df).transform(df)
df_indexed = indexer_kota.fit(df_indexed).transform(df_indexed)

df_indexed.select("kategori", "kategori_idx", "kota", "kota_idx").show(8)

+------------+------------+----------+--------+
|    kategori|kategori_idx|      kota|kota_idx|
+------------+------------+----------+--------+
|     Fashion|         4.0|  Magelang|     1.0|
|Rumah Tangga|         1.0|  Semarang|     2.0|
|  Elektronik|         3.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|  Magelang|     1.0|
|     Makanan|         0.0|  Semarang|     2.0|
|   Kesehatan|         2.0|Yogyakarta|     0.0|
|Rumah Tangga|         1.0|      Solo|     3.0|
|   Kesehatan|         2.0|  Semarang|     2.0|
+------------+------------+----------+--------+
only showing top 8 rows



In [6]:
from pyspark.ml.feature import VectorAssembler

# Menggabungkan seluruh kolom numerik (termasuk hasil StringIndexer) menjadi satu kolom "features"
assembler = VectorAssembler(
    inputCols=["unit_terjual", "harga_satuan", "kategori_idx", "kota_idx", "rating"],
    outputCol="features"
)

df_final = assembler.transform(df_indexed)
df_final.select("features", "pendapatan").show(5, truncate=False)

+---------------------------+------------------+
|features                   |pendapatan        |
+---------------------------+------------------+
|[7.0,150000.0,4.0,1.0,4.0] |997913.0612957058 |
|[4.0,100000.0,1.0,2.0,4.0] |389490.8317158621 |
|[13.0,150000.0,3.0,0.0,4.0]|1936047.6709878189|
|[11.0,100000.0,1.0,1.0,5.0]|1080918.9109179082|
|[8.0,150000.0,0.0,2.0,2.0] |1191696.4943767258|
+---------------------------+------------------+
only showing top 5 rows



In [7]:
# randomSplit membagi data secara acak — 80% untuk latihan (training), 20% untuk pengujian (testing)
train, test = df_final.randomSplit([0.8, 0.2], seed=42)

print(f"Jumlah data training : {train.count()}")
print(f"Jumlah data testing   : {test.count()}")

Jumlah data training : 1612


[Stage 15:>                                                         (0 + 4) / 4]

Jumlah data testing   : 388


In [8]:
from pyspark.ml.regression import LinearRegression

# Mendefinisikan model: featuresCol menunjuk ke kolom fitur, labelCol menunjuk ke kolom target yang diprediksi
lr = LinearRegression(featuresCol="features", labelCol="pendapatan")

# fit() melatih model menggunakan data training -- inilah tahap "pembelajaran" sesungguhnya
model = lr.fit(train)

print("Model berhasil dilatih!")
print("Koefisien tiap fitur:", model.coefficients)
print("Intercept:", model.intercept)

[Stage 19:>                                                         (0 + 4) / 4]

Model berhasil dilatih!
Koefisien tiap fitur: [81316.95453826552,7.643201021360431,1912.9878172114259,4978.004524047301,-2056.7512148209016]
Intercept: -623051.881196131


In [9]:
predictions = model.transform(test)
predictions.select("unit_terjual", "harga_satuan", "kategori", "kota", "pendapatan", "prediction").show(10)

+------------+------------+------------+----------+------------------+-------------------+
|unit_terjual|harga_satuan|    kategori|      kota|        pendapatan|         prediction|
+------------+------------+------------+----------+------------------+-------------------+
|           1|       25000|   Kesehatan|  Magelang| 6863.845479462816| -343907.6721802054|
|           1|       50000|  Elektronik|  Semarang| 49470.32743422343|-150050.15673457773|
|           1|       50000|  Elektronik|      Solo|62393.778432242245|-149185.65464017226|
|           1|       50000|     Makanan|  Magelang| 45646.28680997743| -162823.8759250802|
|           1|       75000|  Elektronik|  Semarang| 80707.98753443373| 41029.868799432996|
|           1|       75000|   Kesehatan|Yogyakarta| 73302.81910021295|  33274.37436376873|
|           1|      100000|     Makanan|      Solo|147035.43677944108| 229292.18419103592|
|           1|      100000|Rumah Tangga|Yogyakarta| 82192.45883479844| 220384.66086574725|

In [10]:
from pyspark.ml.evaluation import RegressionEvaluator

# RMSE (Root Mean Squared Error) -- rata-rata besar kesalahan prediksi, dalam satuan yang sama dengan target (Rupiah)
evaluator_rmse = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="rmse")
rmse = evaluator_rmse.evaluate(predictions)

# R-squared -- seberapa besar proporsi variasi data yang berhasil "dijelaskan" oleh model (0 = buruk, 1 = sempurna)
evaluator_r2 = RegressionEvaluator(labelCol="pendapatan", predictionCol="prediction", metricName="r2")
r2 = evaluator_r2.evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}  ({r2*100:.1f}% variasi data berhasil dijelaskan model)")

[Stage 22:==============>                                           (1 + 3) / 4]

RMSE : Rp 176,339
R²   : 0.8781  (87.8% variasi data berhasil dijelaskan model)


In [11]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
